SQL Server CDC → Databricks Auto Loader/DLT → Claude AI Schema Resolver → Delta Lake (Bronze→Silver→Gold)

In [0]:
# In your Databricks Community Edition notebook

# Get your machine's IP address first:
# Mac/Linux: run "ifconfig" in terminal → look for inet address
# Windows:   run "ipconfig" in cmd → look for IPv4 address

#YOUR_IP = "198.142.152.164"  # ← replace with your actual IP

jdbc_url = "jdbc:postgresql://bore.pub:33627/practice_db"

connection_props = {
    "user":     "admin",
    "password": "admin123",
    "driver":   "org.postgresql.Driver"
}

# Test connection
df_orders = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=connection_props
)

df_orders.show()
#for customer order table 
df_cust = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=connection_props
)

df_cust.show()



print("✅ Connected to Docker PostgreSQL!")

In [0]:
df_cust.printSchema()

In [0]:
df_orders.printSchema()

3. Claude AI Dynamic Mapping Generator
This is the core innovation — Claude analyzes schema drift and generates PySpark transformation code on the fly:

In [0]:
# ============================================================
# READ — pull tables from PostgreSQL
# ============================================================
print("📥 Reading from PostgreSQL...")

orders_df = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=connection_props
)

customers_df = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=connection_props
)

print(f"✅ Orders:    {orders_df.count()} rows — {orders_df.columns}")
print(f"✅ Customers: {customers_df.count()} rows — {customers_df.columns}")

orders_df.show()
customers_df.show()

In [0]:
# ============================================================
# WRITE — fixed version
# ============================================================

def write_to_delta(df, table_name: str):
    silver_table = f"silver_{table_name}"

    print(f"\n💾 Writing {table_name} → {silver_table}")
    print(f"   Rows:    {df.count()}")
    print(f"   Columns: {df.columns}")

    try:
        table_exists = spark.catalog.tableExists(silver_table)

        if table_exists:
            print(f"   🔀 Table exists → dropping and recreating")
            
            # ✅ Fix: drop first then recreate fresh
            spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
            
            (df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(silver_table))

        else:
            print(f"   🆕 Table does not exist → creating fresh")
            (df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(silver_table))

        print(f"   ✅ Successfully written to {silver_table}!")
        
        # Quick verify
        count = spark.read.table(silver_table).count()
        print(f"   ✅ Verified: {count} rows in Delta")

    except Exception as e:
        print(f"   ❌ Failed: {e}")
        raise e


# ============================================================
# RUN
# ============================================================
write_to_delta(orders_df,    "orders")
write_to_delta(customers_df, "customers")

4. Transactional MERGE into Delta Lake

In [0]:
%sql
select * from 

5. Orchestration Pipeline (Delta Live Tables)

In [0]:
# ============================================================
# VERIFY — read back from Delta to confirm
# ============================================================

# ✅ Define TABLES directly in this cell
TABLES = [
    {"table": "orders",    "pks": ["order_id"]},
    {"table": "customers", "pks": ["customer_id"]},
]

print("=" * 60)
print("🔍 VERIFYING DELTA TABLES")
print("=" * 60)

for config in TABLES:
    table_name   = config["table"]
    silver_table = f"silver_{table_name}"

    print(f"\n📋 {silver_table.upper()}")
    print("-" * 40)

    try:
        delta_df = spark.read.table(silver_table)
        print(f"   ✅ Found!")
        print(f"   Rows:    {delta_df.count()}")
        print(f"   Columns: {delta_df.columns}")
        delta_df.show(5, truncate=False)
    except Exception as e:
        print(f"   ❌ Not found: {e}")

print("\n✅ Verification complete!")

Detecing drift before overwriting the changes

In [0]:
# ============================================================
# DETECT DRIFT
# ============================================================
import json

# Check orders drift
orders_drift = detector.detect_drift("orders", orders_df)
print("📊 Orders Drift Report:")
print(json.dumps(orders_drift, indent=2))

# Check customers drift
customers_drift = detector.detect_drift("customers", customers_df)
print("\n📊 Customers Drift Report:")
print(json.dumps(customers_drift, indent=2))

In [0]:
# ============================================================
# SIMULATE DRIFT
# Step 1: Run this SQL in DBeaver first:
#
# ALTER TABLE orders ADD COLUMN discount_pct DECIMAL(5,2) DEFAULT 0.0;
# UPDATE orders SET discount_pct = 10.0 WHERE order_id = 1;
#
# Step 2: Then run this cell
# ============================================================

print("Re-reading orders after schema change...")

orders_df_new = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=connection_props
)

print(f"New columns: {orders_df_new.columns}")
# You should now see discount_pct in the list!

# Write to Delta — mergeSchema handles the new column
write_to_delta(orders_df_new, "orders")

# Verify new column is in Delta
delta_df = spark.read.table("silver_orders")
print(f"\nDelta columns now: {delta_df.columns}")
delta_df.show(5, truncate=False)

| Concern              | Solution                                                                                           |
|----------------------|----------------------------------------------------------------------------------------------------|
| Schema drift         | Claude generates PySpark mapping at runtime — no manual pipeline edits                             |
| Transactional integrity | Delta MERGE with __$operation semantics maps exactly to SQL Server CDC ops                      |
| Dropped columns      | Historical columns preserved as null in Delta — never physically removed                           |
| Type changes         | try_cast() with null fallback prevents row-level failures from poisoning batches                   |
| Audit trail          | Every drift event logged to audit.schema_drift_log with full Claude-generated diff                 |
| Cost control         | Claude only called when has_drift = True, not on every batch                                       |